<a href="https://colab.research.google.com/github/SMC-AAU-CPH/ML-For-Beginners/blob/main/7-EdgeImpulse/MLME24_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import tensorflow as tf
import numpy as np
import os

# Tutorial 1: Continous Motion Recognition
The project [https://studio.edgeimpulse.com/public/14299/latest](https://studio.edgeimpulse.com/public/14299/latest)

In [21]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=412593
DSP_ID=31
LRN_ID=35

# if there is no data1 subfolder, create one and get the data inside it; -q flag is for quiet mode
if not os.path.exists('data1'):
    os.makedirs('data1')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data1/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data1/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data1/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data1/Y_split_test.npy

In [22]:
# THE MODEL: download only if model1/model.h5 is missing; unzip -o overwrites without prompting
if not os.path.exists('model1/model.h5'):
    os.makedirs('model1', exist_ok=True)
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-h5 -O model1/model1.zip
    !cd model1 && unzip -q -o model1.zip && rm model1.zip

model = tf.keras.models.load_model('model1/model.h5')

In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 20)             │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_pred (Dense)                  │ (None, 4)              │            44 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,056 (4.13 KB)

 Trainable params: 1,054 (4.12 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [24]:
test_data = np.load('data1/X_split_test.npy')
test_labels = np.load('data1/Y_split_test.npy')

In [25]:
# The labels are in the first column and go like "idle", "snake", "updown", "wave, and "anomaly" or "other"?
test_labels[:,0]

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 5, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [26]:
predictions = model.predict(test_data)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [27]:
np.argmax(predictions, axis = 1)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [28]:
# For calculating the accuracy we subtract -2 from the test labels (labels 2-5 are the model's 4 classes); label 1 is not a model class, so it is skipped
keep = test_labels[:,0] >= 2
accuracy = np.mean(np.argmax(predictions[keep], axis = 1) == (test_labels[keep,0]-2))
print(f'Accuracy: {accuracy * 100: .2f}%')

Accuracy:  99.80%


## Alternative eval

`model.compile_metrics` on the same samples as the accuracy above (labels 2–5). The cell below prints both numbers and checks that they are equal.

| Metric | Samples | Accuracy |
|---|---|---|
| Formula (above) | 489 | 99.80% |
| `compile_metrics` (below) | 489 | 99.80% |

In [29]:
# compile_metrics on the same samples as the accuracy formula above (labels 2-5 map to classes 0-3)
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
keep = test_labels[:, 0] >= 2
loss, acc = model.evaluate(test_data[keep], test_labels[keep, 0] - 2, verbose=0)
print(f'compile_metrics accuracy: {acc * 100: .2f}% (loss {loss:.4f})')
print(f'Formula accuracy:         {accuracy * 100: .2f}%')
print('Equal:', np.isclose(acc, accuracy))

compile_metrics accuracy:  99.80% (loss 0.0119)
Formula accuracy:          99.80%
Equal: True


# Tutorial 2: Responding to Voice

In [30]:
# The data and model are to be taken from the project page https://studio.edgeimpulse.com/public/14225/latest
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=412592
DSP_ID=25
LRN_ID=5

# if there is no data subfolder, create one, and get the data inside it; -q flag is for quiet mode

if not os.path.exists('data'):
    os.makedirs('data')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data/Y_split_test.npy


In [31]:
# THE MODEL
!wget https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-h5 -O model2.zip

# unzip quitely and remove model2.zip
!unzip -q model2.zip
!rm model2.zip
!ls

--2026-10-09 16:17:22--  https://studio.edgeimpulse.com/v1/api/412592/learn-data/5/model/tflite-h5
Resolving studio.edgeimpulse.com (studio.edgeimpulse.com)... 3.174.18.75, 3.174.18.108, 3.174.18.14, ...
Connecting to studio.edgeimpulse.com (studio.edgeimpulse.com)|3.174.18.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/octet-stream]
Saving to: ‘model2.zip’

model2.zip              [ <=>                ]  16.00K  --.-KB/s    in 0s      

2026-10-09 16:17:23 (38.4 MB/s) - ‘model2.zip’ saved [16381]

Copy of 07_dimensionality_reduction.ipynb
Copy of 08_unsupervised_learning.ipynb
MLME26_Assignment_2.ipynb
Portal-Removals-261003.org
Session5
__pycache__
courseid_60493_participants.csv
data
data1
invite_users_to_organization.py
invite_users_to_project.py
model.h5
model1
my_ladybug.png
my_mnist.mmap
org_users_last_access.csv
project_summary.py
prune_inactive_users.py
skills-lock.json


In [32]:
model = tf.keras.models.load_model('model.h5')

/Users/cer/mamba/lib/python3.12/site-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [33]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gaussian_noise (GaussianNoise)  │ (None, 624)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 48, 13, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 48, 13, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 7, 8)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 7, 8)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 7, 16)      │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 4, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 4, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_pred (Dense)                  │ (None, 3)              │         2,307 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,557 (13.90 KB)

 Trainable params: 3,555 (13.89 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [34]:
test_data = np.load('data/X_split_test.npy')
test_labels = np.load('data/Y_split_test.npy')

In [35]:
# The labels are in the first column and go like 1=helloworld, 2=noise, 3=unknown
test_labels[:,0]

array([1, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1,

In [36]:
predictions = model.predict(test_data)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [37]:
np.argmax(predictions, axis = 1)

array([0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 0, 2, 2, 2, 2, 2, 0, 0, 1, 2, 2, 1, 2, 0, 0, 2, 2, 2, 2,
       1, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 0, 2, 2,
       0, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 1, 2, 2, 1, 2, 0, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 1, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,

In [38]:
# For calculating the accuracy we substract -1
test_labels[:,0]-1

array([0, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,

In [39]:
accuracy = np.mean(np.argmax(predictions, axis = 1) == (test_labels[:,0]-1))

In [40]:
print(f'Accuracy: {accuracy * 100: .2f}%')

Accuracy:  90.80%


## Alternative eval

`model.compile_metrics` on the full test set. This matches the notebook's accuracy above.

| Metric | Accuracy |
|---|---|
| Notebook formula (above) | 90.80% |
| `compile_metrics` | 90.80% |

In [41]:
# compile_metrics on all test samples (labels 1-3 map to classes 0-2)
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
loss, acc = model.evaluate(test_data, test_labels[:, 0] - 1, verbose=0)
print(f'compile_metrics accuracy: {acc * 100:.2f}% (loss {loss:.4f})')

compile_metrics accuracy: 90.80% (loss 0.2789)


# Tutorial 3: Recognize sounds from audio
https://studio.edgeimpulse.com/public/14301/latest

In [42]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=417225
DSP_ID=24
LRN_ID=28

# if there is no data subfolder, create one, and get the data inside it; -q flag is for quiet mode

if not os.path.exists('data3'):
    os.makedirs('data3')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data3/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data3/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data3/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data3/Y_split_test.npy


In [43]:
# THE MODEL
if not os.path.exists('model3'):
    os.makedirs('model3')
!wget https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-h5 -O model3/model3.zip

# unzip model1.zip quitely as model1.h5 and remove model1.zip
!cd model3 && unzip -q model3.zip
!rm model3/model3.zip
!cd ..

--2026-10-09 16:18:36--  https://studio.edgeimpulse.com/v1/api/417225/learn-data/28/model/tflite-h5
Resolving studio.edgeimpulse.com (studio.edgeimpulse.com)... 3.174.18.75, 3.174.18.108, 3.174.18.14, ...
Connecting to studio.edgeimpulse.com (studio.edgeimpulse.com)|3.174.18.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/octet-stream]
Saving to: ‘model3/model3.zip’

model3/model3.zip       [ <=>                ]  12.99K  --.-KB/s    in 0s      

2026-10-09 16:18:36 (3.10 GB/s) - ‘model3/model3.zip’ saved [13298]



In [44]:
model = tf.keras.models.load_model('model3/model.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 99, 65)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 99, 8)          │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 50, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 16)         │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_pred (Dense)                  │ (None, 2)              │           802 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,772 (10.83 KB)

 Trainable params: 2,770 (10.82 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [45]:
test_data = np.load('data3/X_split_test.npy')
test_labels = np.load('data3/Y_split_test.npy')

In [46]:
test_labels[:,0]

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [47]:
predictions = model.predict(test_data)
np.argmax(predictions, axis = 1)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [48]:
accuracy = np.mean(np.argmax(predictions, axis = 1) == (test_labels[:,0]-1))
print(f'Accuracy: {accuracy * 100: .2f}%')

Accuracy:  100.00%


## Alternative eval

`model.compile_metrics` on the full test set. This matches the notebook's accuracy above.

| Metric | Accuracy |
|---|---|
| Notebook formula (above) | 100.00% |
| `compile_metrics` | 100.00% |

In [49]:
# compile_metrics on all test samples (labels 1-2 map to classes 0-1)
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
loss, acc = model.evaluate(test_data, test_labels[:, 0] - 1, verbose=0)
print(f'compile_metrics accuracy: {acc * 100:.2f}% (loss {loss:.4f})')

compile_metrics accuracy: 100.00% (loss 0.0000)


# Tutorial 4: Adding sight to your sensors
https://studio.edgeimpulse.com/public/14227/latest

The original .h5 model did not work with tensorflow==2.17.0, so this tutorial now loads the TensorFlow SavedModel export instead. 
Tested on tensorflow==2.20.0, as Tutorials 1–3.

**Sensor & Block Information**
- Image data
- Image DSP block for normalizing image files
- Transfer Learning block with prediction outputs: "lamp", "plant", "unknown"

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=792097
DSP_ID=31
LRN_ID=19

# if there is no data subfolder, create one, and get the data inside it; -q flag is for quiet mode

if not os.path.exists('data4'):
    os.makedirs('data4')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data4/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data4/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data4/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data4/Y_split_test.npy


In [ ]:
# SavedModel export of the same model (works with TensorFlow 2.20)
if not os.path.exists('model4'):
    os.makedirs('model4')
!wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-savedmodel -O model4/model4.zip
!cd model4 && unzip -q -o model4.zip && rm model4.zip
!cd ..

In [ ]:
# load the SavedModel and its serving_default signature
model4 = tf.saved_model.load('model4/saved_model')
infer = model4.signatures['serving_default']

In [ ]:
test_data = np.load('data4/X_split_test.npy')
test_labels = np.load('data4/Y_split_test.npy')
test_labels[:,0]

In [ ]:
# Reshape test_data to the model's input shape (96 x 96 x 3 images)
test_data = test_data.reshape((-1, 96, 96, 3)).astype(np.float32)

# Run inference in one batch; the output key is 'output_0'
prediction_results = infer(tf.constant(test_data))['output_0'].numpy()

np.argmax(prediction_results, axis = 1)

In [ ]:
# calculate accuracy
predicted_indices = np.argmax(prediction_results, axis = 1)
accuracy = np.mean(predicted_indices == (test_labels[:,0]-1))
print(f'Accuracy: {accuracy * 100: .2f}%')

## Alternative eval

The SavedModel isn't a Keras model, so `compile_metrics` is only available after wrapping it in a Keras `Model`. This wrapper uses `TFSMLayer` to call the SavedModel.

| Metric | Accuracy |
|---|---|
| Notebook formula (above) | 85.45% |
| `compile_metrics` (wrapped SavedModel) | 85.45% |

In [ ]:
# Wrap the SavedModel's serving_default output in a Keras model so compile_metrics can run
inp = tf.keras.Input((96, 96, 3))
wrapped = tf.keras.Model(inp, tf.keras.layers.TFSMLayer('model4/saved_model', call_endpoint='serving_default')(inp)['output_0'])
wrapped.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# the SavedModel signature has a fixed batch size of 1
loss, acc = wrapped.evaluate(test_data.reshape((-1, 96, 96, 3)).astype(np.float32), test_labels[:, 0] - 1,
                             verbose=0, batch_size=1)
print(f'compile_metrics accuracy: {acc * 100:.2f}% (loss {loss:.4f})')

# Tutorial 5A: Object Detection
https://studio.edgeimpulse.com/public/25483/latest

The SavedModel export of this model returns all-zero outputs for any input, so this tutorial uses the **TFLite int8** export instead. Labels are JSON bounding boxes (`x, y, w, h` in pixels of the 320x320 image), not a numeric array.

In [ ]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=408972
DSP_ID=3
LRN_ID=5

# if there is no data subfolder, create one, and get the data inside it; -q flag is for quiet mode

if not os.path.exists('data5a'):
    os.makedirs('data5a')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data5a/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data5a/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data5a/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data5a/Y_split_test.npy


In [ ]:
# TFLite (int8) export of the same model
if not os.path.exists('model5a'):
    os.makedirs('model5a')
!wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-int8 -O model5a/model5a.tflite

In [ ]:
import json
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path='model5a/model5a.tflite')
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]

# TFLite SSD post-processing outputs; names as exported by Edge Impulse
outs = {'count': 'StatefulPartitionedCall:0', 'classes': 'StatefulPartitionedCall:2',
        'boxes': 'StatefulPartitionedCall:3', 'scores': 'StatefulPartitionedCall:1'}
outs = {k: next(d for d in interpreter.get_output_details() if d['name'] == v) for k, v in outs.items()}
print('input', inp['shape'], inp['dtype'].__name__)

In [ ]:
test_data = np.load('data5a/X_split_test.npy')

In [ ]:
# The labels are JSON bounding boxes, so parse the file as JSON (not np.load)
with open('data5a/Y_split_test.npy', 'rb') as f:
    data = json.loads(f.read().decode())
print(len(data['samples']), 'test samples')

In [ ]:
# Extract the ground-truth labels and boxes for each test sample
labels = []
for sample in data['samples']:
    labels.append([(b['label'], b['x'], b['y'], b['w'], b['h']) for b in sample['boundingBoxes']])
print(labels[0])

In [ ]:
# Quantize the float image to int8, run the model, and return detections in pixels
def detect(image, threshold=0.3):
    s, z = inp['quantization']
    x = np.clip(np.round(image.reshape(1, 320, 320, 3) / s + z), -128, 127).astype(np.int8)
    interpreter.set_tensor(inp['index'], x)
    interpreter.invoke()
    get = lambda k: interpreter.get_tensor(outs[k]['index'])[0]
    scores, boxes, classes = get('scores'), get('boxes'), get('classes').astype(int)
    keep = scores >= threshold
    # boxes are [ymin, xmin, ymax, xmax] in 0-1; class index + 1 is the label
    return [(int(c) + 1, b[1] * 320, b[0] * 320, (b[3] - b[1]) * 320, (b[2] - b[0]) * 320, float(s))
            for b, c, s in zip(boxes[keep], classes[keep], scores[keep])]

In [ ]:
predictions = [detect(img) for img in test_data]
for i in range(3):
    print('GT  ', labels[i])
    print('pred', [(d[0], *[round(v) for v in d[1:5]], round(d[5], 2)) for d in predictions[i]])

In [ ]:
# Match detections to ground truth by label and IoU >= 0.5
def iou(a, b):
    iw = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union else 0

tp = fp = fn = 0
for gts, dets in zip(labels, predictions):
    matched = set()
    for d in dets:
        best = None
        for gi, (gl, *gbox) in enumerate(gts):
            if gi in matched or gl != d[0]:
                continue
            v = iou(d[1:5], gbox)
            if v >= 0.5 and (best is None or v > best[1]):
                best = (gi, v)
        if best:
            tp += 1
            matched.add(best[0])
        else:
            fp += 1
    fn += len(gts) - len(matched)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print(f'IoU>=0.5, score>=0.3: TP {tp}  FP {fp}  FN {fn}  precision {precision:.2f}  recall {recall:.2f}')

## Alternative eval

`compile_metrics` doesn't apply to this model. It's a TFLite detector with no Keras model, so there's no compiled metric to call. The closest equivalent is label accuracy on the boxes that match a ground-truth box (IoU ≥ 0.5, score ≥ 0.3).

| Metric | Result |
|---|---|
| Matched boxes with the correct label | 19 / 19 = 100.00% |
| Recall (from the notebook's evaluation above) | 1.00 |

In [ ]:
# Label accuracy on detections that match a ground-truth box (IoU >= 0.5); predictions already use score >= 0.3
correct = matched = 0
for gts, dets in zip(labels, predictions):
    used = set()
    for d in dets:
        best = None
        for gi, (gl, *gb) in enumerate(gts):
            if gi in used:
                continue
            v = iou(d[1:5], gb)
            if v >= 0.5 and (best is None or v > best[1]):
                best = (gi, v, gl)
        if best:
            used.add(best[0])
            matched += 1
            correct += int(best[2] == d[0])
print(f'Label accuracy on matched boxes: {correct}/{matched} = {correct / matched * 100:.2f}%')

# TODO Tutorial 5B: Object detection FOMO
https://studio.edgeimpulse.com/public/90393/latest


TODO: the label data must differ being bounding box. Check how to load!!!

TODO: add README



In [ ]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=90395
DSP_ID=3
LRN_ID=5

# if there is no data subfolder, create one, and get the data inside it; -q flag is for quiet mode

if not os.path.exists('data5b'):
    os.makedirs('data5b')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/training -O data5b/X_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/training -O data5b/Y_split_train.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data5b/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data5b/Y_split_test.npy


In [ ]:
# THE MODEL
if not os.path.exists('model5b'):
    os.makedirs('model5b')
!wget https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-h5 -O model5b/model5b.zip

# unzip model1.zip quitely as model1.h5 and remove model1.zip
!cd model5b && unzip -q model5b.zip
!rm model5b/model5b.zip
!cd ..

In [ ]:
model = tf.keras.models.load_model('model5b/model.h5')
model.summary()

In [ ]:
test_data = np.load('data5b/X_split_test.npy')


## Alternative eval

Not run. Tutorial 5B uses project `90395`, which no longer exists, and project `711906` is private and needs its own API key. There's no test data or model to evaluate yet.

# Tutorial 5C: FOMO CarPark
https://studio.edgeimpulse.com/studio/1134048

The Keras `.h5` export of this model does not load on tensorflow==2.20.0 (`DepthwiseConv2D` config error), so this tutorial uses the **TFLite int8** export.

FOMO outputs a 12x12 grid of class scores for the 96x96 input (8 px per cell). Channel 0 is background and channel 1 is car. The test set also has 3 boxes with label 2, which is not an output of this model, so they are reported but not scored.

In [ ]:
# no cloning needed, the following are extracted from the project page dashboard for auto downloads
PROJECT_ID=1134048
DSP_ID=3
LRN_ID=5

if not os.path.exists('data5c'):
    os.makedirs('data5c')
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/x/testing -O data5c/X_split_test.npy
    !wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/dsp-data/$DSP_ID/y/testing -O data5c/Y_split_test.npy

In [ ]:
# TFLite (int8) export; the Keras h5 export does not load on tensorflow 2.20
if not os.path.exists('model5c'):
    os.makedirs('model5c')
!wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-int8 -O model5c/model5c.tflite

In [ ]:
import json
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path='model5c/model5c.tflite')
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print('input', inp['shape'], inp['dtype'].__name__, '| output', out['shape'], out['dtype'].__name__)

In [ ]:
# The labels are JSON bounding boxes, so parse the file as JSON (not np.load)
test_data = np.load('data5c/X_split_test.npy')
with open('data5c/Y_split_test.npy', 'rb') as f:
    data = json.loads(f.read().decode())
print(len(data['samples']), 'test samples')

In [ ]:
CELL = 8          # 96 px input / 12 grid cells
THRESHOLD = 0.5

def predict_grid(image):
    """Return the 12x12 car probability grid for one 96x96 grayscale image."""
    s, z = inp['quantization']
    x = np.clip(np.round(image.reshape(1, 96, 96, 1) / s + z), -128, 127).astype(np.int8)
    interpreter.set_tensor(inp['index'], x)
    interpreter.invoke()
    so, zo = out['quantization']
    probs = (interpreter.get_tensor(out['index'])[0].astype(np.float32) - zo) * so
    return probs[:, :, 1]   # channel 0 = background, channel 1 = car

grids = [predict_grid(img) for img in test_data]

In [ ]:
# A detection is a grid cell above THRESHOLD that is the maximum of its 3x3 neighbourhood
def detections(grid):
    cells = []
    for r in range(12):
        for c in range(12):
            v = grid[r, c]
            if v >= THRESHOLD and v >= grid[max(r - 1, 0):r + 2, max(c - 1, 0):c + 2].max():
                cells.append((r, c))
    return set(cells)

# Ground truth: the grid cell that contains each box centre (label 1 only)
tp = fp = fn = 0
for sample, grid in zip(data['samples'], grids):
    gt_cells = set()
    for b in sample['boundingBoxes']:
        if b['label'] != 1:
            continue
        cx, cy = b['x'] + b['w'] / 2, b['y'] + b['h'] / 2
        gt_cells.add((min(int(cy // CELL), 11), min(int(cx // CELL), 11)))
    det = detections(grid)
    tp += len(det & gt_cells)
    fp += len(det - gt_cells)
    fn += len(gt_cells - det)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print(f'Label 1 (car), threshold {THRESHOLD}: TP {tp}  FP {fp}  FN {fn}  precision {precision:.2f}  recall {recall:.2f}')
print('Label 2 boxes (not a model class, not scored):',
      sum(b['label'] == 2 for s in data['samples'] for b in s['boundingBoxes']))

## Alternative eval

The Keras `.h5` model loads only with a small shim: `DepthwiseConv2D` in this export has a `groups` argument that TensorFlow 2.20 doesn't accept. `compile_metrics` is then run per grid cell (0 = background, 1 = car). That's dominated by background cells, so it's a weaker number than the detection-level result.

| Metric | Result |
|---|---|
| `compile_metrics`, per-cell accuracy (float h5) | 96.77% |
| Detections, float h5 (TP / FP / FN) | 346 / 41 / 83, precision 0.89, recall 0.81 |
| Detections, int8 TFLite (above) | 370 / 47 / 59, precision 0.89, recall 0.86 |

In [ ]:
# Keras model from the float .h5 export (needs the groups shim), then compile_metrics on per-cell labels
class NoGroupsDepthwiseConv2D(tf.keras.layers.DepthwiseConv2D):
    def __init__(self, *args, groups=1, **kwargs):   # groups is always 1 for depthwise convs
        super().__init__(*args, **kwargs)

!wget -q https://studio.edgeimpulse.com/v1/api/$PROJECT_ID/learn-data/$LRN_ID/model/tflite-h5 -O model5c/model5c_h5.zip
!cd model5c && unzip -q -o model5c_h5.zip && rm model5c_h5.zip
keras_model = tf.keras.models.load_model('model5c/model.h5', compile=False,
                                         custom_objects={'DepthwiseConv2D': NoGroupsDepthwiseConv2D})

# per-cell labels: 1 where a label-1 box centre falls in that grid cell, else 0
cell_y = np.zeros((len(data['samples']), 12, 12), np.int32)
for n, s in enumerate(data['samples']):
    for b in s['boundingBoxes']:
        if b['label'] == 1:
            cx, cy = b['x'] + b['w'] / 2, b['y'] + b['h'] / 2
            cell_y[n, min(int(cy // CELL), 11), min(int(cx // CELL), 11)] = 1

keras_model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
loss, acc = keras_model.evaluate(test_data.reshape((-1, 96, 96, 1)).astype(np.float32), cell_y, verbose=0)
print(f'compile_metrics per-cell accuracy: {acc * 100:.2f}% (loss {loss:.4f})')